<a href="https://colab.research.google.com/github/Sosina47/aws-sbg-tasks/blob/main/week2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import pandas as pd

faqs = [
    {
        "question": "How do I reset my password?",
        "answer": "Visit the password reset page and follow the instructions."
    },
    {
        "question": "How can I change my account password?",
        "answer": "Open account settings and choose the password option."
    },
    {
        "question": "How do I apply for university admission?",
        "answer": "Complete the online application form and submit your documents."
    },
    {
        "question": "When does course registration open?",
        "answer": "Check the academic calendar for registration dates."
    },
    {
        "question": "What should I do if I cannot log in?",
        "answer": "Check your credentials or use the account recovery process."
    }
]

df = pd.DataFrame(faqs)

df

,question,answer
0,How do I reset my password?,Visit the password reset page and follow the i...
1,How can I change my account password?,Open account settings and choose the password ...
2,How do I apply for university admission?,Complete the online application form and submi...
3,When does course registration open?,Check the academic calendar for registration d...
4,What should I do if I cannot log in?,Check your credentials or use the account reco...


In [16]:
print("Dataset shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nMissing values:")
print(df.isnull().sum())

Dataset shape: (5, 2)

Column names: ['question', 'answer']

Missing values:
question    0
answer      0
dtype: int64


In [ ]:
df.to_csv("faqs.csv", index=False)

df = pd.read_csv("faqs.csv")

df.head()

In [ ]:

# Make a copy so we keep the original dataset unchanged.
clean_df = df.copy()

# Clean the question column.
clean_df["question"] = (
    clean_df["question"]
    .str.lower()                 # Convert text to lowercase.
    .str.strip()                 # Remove spaces at the beginning and end.
    .str.replace(r"[^\w\s]", "", regex=True)  # Remove punctuation.
    .str.replace(r"\s+", " ", regex=True)     # Replace repeated whitespace with one space.
)

# Clean the answer column using the same operations.
clean_df["answer"] = (
    clean_df["answer"]
    .str.lower()
    .str.strip()
    .str.replace(r"[^\w\s]", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
)

# Display the cleaned dataset.
clean_df


In [ ]:
print("Original first question:")
print(df["question"].iloc[0])

print("\nCleaned first question:")
print(clean_df["question"].iloc[0])

print("\nMissing values after cleaning:")
print(clean_df.isnull().sum())


In [ ]:

# Combine each question and answer into one searchable text.
clean_df["text"] = (
    "Question: " + clean_df["question"]
    + " Answer: " + clean_df["answer"]
)

# Display the question, answer, and combined text.
clean_df[["question", "answer", "text"]]


In [ ]:
chunks = clean_df["text"].tolist()

chunk_ids = [f"faq_{i}" for i in range(len(chunks))]

for chunk_id, chunk in zip(chunk_ids, chunks):
    print(f"{chunk_id}: {chunk}\n")

In [22]:

%pip install -q sentence-transformers


In [ ]:

from sentence_transformers import SentenceTransformer

# Load a small, pretrained model for semantic search.
model = SentenceTransformer("all-MiniLM-L6-v2")


In [ ]:
embeddings = model.encode(chunks)

print("Number of chunks:", len(chunks))
print("Embedding shape:", embeddings.shape)
print("First embedding:")
print(embeddings[0])

In [25]:
%pip install -q chromadb

In [ ]:

import chromadb

print("Chroma version:", chromadb.__version__)

In [ ]:
import chromadb

# Create an in-memory Chroma database.
client = chromadb.Client()

# Create a collection to hold our FAQ data.
collection = client.get_or_create_collection(
    name="faq_collection"
)

print("Collection created successfully!")

In [ ]:

if collection.count() == 0:
    collection.add(
        ids=chunk_ids,
        documents=chunks,
        embeddings=embeddings.tolist()
    )
    print("FAQs added successfully!")
else:
    print("FAQs are already stored; skipping duplicate insertion.")

print("Total FAQs stored:", collection.count())

In [ ]:
query = "I forgot my password. How can I get back into my account?"

# Convert the query into an embedding using the same model.
query_embedding = model.encode(query).tolist()

print("Query:", query)
print("Number of values in query embedding:", len(query_embedding))


In [34]:

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

for i in range(3):
    print(f"\nMatch {i + 1}")
    print("FAQ ID:", results["ids"][0][i])
    print("Text:", results["documents"][0][i])
    print("Distance:", results["distances"][0][i])


Match 1
FAQ ID: faq_4
Text: Question: what should i do if i cannot log in Answer: check your credentials or use the account recovery process
Distance: 0.6709061861038208

Match 2
FAQ ID: faq_0
Text: Question: how do i reset my password Answer: visit the password reset page and follow the instructions
Distance: 0.7289966940879822

Match 3
FAQ ID: faq_1
Text: Question: how can i change my account password Answer: open account settings and choose the password option
Distance: 0.85322105884552
